In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold,train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder,MinMaxScaler
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_selector,make_column_transformer

In [2]:
df= pd.read_csv("F:\MACHINE LEARNING\SOMETHING\Datasets\Sacremento.csv")
df.head(4)

,city,zip,beds,baths,sqft,type,price,latitude,longitude
0,SACRAMENTO,z95838,2,1.0,836,Residential,59222,38.631913,-121.434879
1,SACRAMENTO,z95823,3,1.0,1167,Residential,68212,38.478902,-121.431028
2,SACRAMENTO,z95815,2,1.0,796,Residential,68880,38.618305,-121.443839
3,SACRAMENTO,z95815,2,1.0,852,Residential,69307,38.616835,-121.439146


In [3]:
df.isnull().sum()

city         0
zip          0
beds         0
baths        0
sqft         0
type         0
price        0
latitude     0
longitude    0
dtype: int64

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 932 entries, 0 to 931
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   city       932 non-null    object 
 1   zip        932 non-null    object 
 2   beds       932 non-null    int64  
 3   baths      932 non-null    float64
 4   sqft       932 non-null    int64  
 5   type       932 non-null    object 
 6   price      932 non-null    int64  
 7   latitude   932 non-null    float64
 8   longitude  932 non-null    float64
dtypes: float64(3), int64(3), object(3)
memory usage: 65.7+ KB


In [5]:
df.drop('zip',axis=1,inplace=True)

In [6]:
X,y=df.drop('price',axis=1),df.price

In [21]:
ohe=OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore').set_output(transform='pandas')
sc=StandardScaler(with_mean=False).set_output(transform='pandas')

ct=make_column_transformer(
    (ohe,make_column_selector(dtype_include=object)),
    ('passthrough',make_column_selector(dtype_exclude=object)),
      verbose_feature_names_out=False).set_output(transform='pandas')


In [22]:
Elast=ElasticNet()
Rf=RandomForestRegressor(random_state=26)
lgm=LGBMRegressor()
xgb=XGBRegressor()

In [23]:
pipe1=Pipeline([('ct',ct),('sc',sc),('elast',Elast)])
pipe1.get_params()

{'memory': None,
 'steps': [('ct',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  handle_unknown='ignore',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000244B6FC91F0>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000244B6FCAE10>)],
                     verbose_feature_names_out=False)),
  ('sc', StandardScaler(with_mean=False)),
  ('elast', ElasticNet())],
 'transform_input': None,
 'verbose': False,
 'ct': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                handle_unknown='ignore',
                  

In [24]:
kv= KFold(n_splits=5, shuffle=True, random_state=26)
params={
    'elast__l1_ratio':[0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0],
        'elast__alpha':[0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]}
gcv1=GridSearchCV(pipe1,param_grid=params,scoring='r2',n_jobs=-1,cv=kv)
gcv1.fit(X,y)
print(gcv1.best_score_)
print(gcv1.best_params_)


0.6607054875546629
{'elast__alpha': 0.1, 'elast__l1_ratio': 0.7}


In [25]:
pipe2=Pipeline([('ct',ct),('sc',sc),('rf',Rf)])
pipe2.get_params()

{'memory': None,
 'steps': [('ct',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  handle_unknown='ignore',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000244B6FC91F0>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000244B6FCAE10>)],
                     verbose_feature_names_out=False)),
  ('sc', StandardScaler(with_mean=False)),
  ('rf', RandomForestRegressor(random_state=26))],
 'transform_input': None,
 'verbose': False,
 'ct': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                handle_unknown='igno

In [29]:
kv= KFold(n_splits=5, shuffle=True, random_state=26)
params={
        'rf__max_depth': np.arange(1,20),
        'rf__n_estimators':[10,30,50,75,100]}
gcv2=GridSearchCV(pipe2,param_grid=params,scoring='r2',n_jobs=-1,cv=kv)
gcv2.fit(X,y)
print(gcv2.best_score_)
print(gcv2.best_params_)

0.6560879881146408
{'rf__max_depth': np.int64(10), 'rf__n_estimators': 100}


In [27]:
pipe3=Pipeline([('ct',ct),('sc',sc),('lgb',lgm)])
pipe3.get_params()

{'memory': None,
 'steps': [('ct',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  handle_unknown='ignore',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000244B6FC91F0>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x00000244B6FCAE10>)],
                     verbose_feature_names_out=False)),
  ('sc', StandardScaler(with_mean=False)),
  ('lgb', LGBMRegressor())],
 'transform_input': None,
 'verbose': False,
 'ct': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                handle_unknown='ignore',
                 

In [28]:
kv= KFold(n_splits=5, shuffle=True, random_state=26)
params={}
        
gcv3=GridSearchCV(pipe3,param_grid=params,scoring='r2',n_jobs=-1,cv=kv)
gcv3.fit(X,y)
print(gcv3.best_score_)
print(gcv3.best_params_)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000220 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 804
[LightGBM] [Info] Number of data points in the train set: 932, number of used features: 16
[LightGBM] [Info] Start training from score 246661.583691
0.643538073450416
{}
